# 第 12 章 · 前缀和 scan：每个位置都要"前面所有数的和"

本章回答：前缀和看起来每一步都依赖上一步，怎么并行？Hillis–Steele 和 Blelloch 两种算法差在哪？block 内用 shuffle 怎么做？跨 block 的 scan 为什么麻烦？scan 在 AI kernel 里用在哪？

配套讲义：仓库里 `cuda-programming/notes/lessons/12-scan.md`（讲解、动画、练习都在讲义里）。对应原文：Oxford lec4 第 20–33 页（`materials/text/oxford-lec4.txt` 第 413–650 行：定义、局部 scan 两个版本、全局 scan 两种做法）；PG §4.4.6.2 Scans（`text/cuda-programming-guide.txt` 第 15546–15563 行，cooperative groups 的 scan）；PMPP 第 5 版 ch.11 Prefix sum；Stanford CS149 asst3（`courses/stanford-cs149/asst3/`）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 定义 | inclusive / exclusive | `inclusive(v)`、`exclusive(v)` | 3 4 8 9 14 23 25 31 / 0 3 4 8 9 14 23 25 |
| 第二步 Hillis–Steele | 每轮加左边隔 d 个 | `hillis_steele(v)` | 三轮结果同讲义表；17 次加法 |
| 第三步 Blelloch | 上扫 + 下扫 | `blelloch(v)` | exclusive 结果；14 次加法 |
| 第四步 warp 内 scan | `__shfl_up_sync` | `warp_scan(np.array(v))` | = inclusive |
| 第五步 三段式 | 局部 scan → 总和 scan → 加回去 | `three_phase_scan(big, 8)` | 和 `np.cumsum` 一样 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `blelloch`**（来自 `ch12_scan.py`）：上扫（保留中间和的树形归约）+ 下扫（根置 0，左传自己、右传自己 + 左），得到 exclusive scan。

In [ ]:
def blelloch(v) -> tuple[list[int], int]:
    """上扫（保留中间和的树形归约）+ 下扫（根置 0，左传自己、右传自己 + 左），得到 exclusive scan。"""
    temp, n, adds = list(v), len(v), 0
    d = 1
    while d < n:                                         # 上扫
        for t in range(2 * d - 1, n, 2 * d):
            temp[t] += temp[t - d]
            adds += 1
        d *= 2
    temp[-1] = 0
    d = n // 2
    while d >= 1:                                        # 下扫
        for t in range(2 * d - 1, n, 2 * d):
            temp[t - d], temp[t] = temp[t], temp[t] + temp[t - d]
            adds += 1
        d //= 2
    return temp, adds

**函数 `exclusive`**（来自 `ch12_scan.py`）：v_j = 前面（不含自己）之和，v_0 = 0。

In [ ]:
def exclusive(v) -> np.ndarray:
    """v_j = 前面（不含自己）之和，v_0 = 0。"""
    return np.concatenate([[0], np.cumsum(v)[:-1]])

**函数 `hillis_steele`**（来自 `ch12_scan.py`）：每轮 temp[t] += temp[t − d]（t ≥ d），d = 1, 2, 4…。先全员读完再写（两道栅栏）。返回每轮结果和加法次数。

In [ ]:
def hillis_steele(v) -> tuple[list[list[int]], int]:
    """每轮 temp[t] += temp[t − d]（t ≥ d），d = 1, 2, 4…。先全员读完再写（两道栅栏）。返回每轮结果和加法次数。"""
    temp = list(v)
    rows, adds, d = [], 0, 1
    while d < len(temp):
        old = temp[:]                                   # 第一道栅栏之后：所有人先读旧值
        temp = [old[t] + (old[t - d] if t >= d else 0) for t in range(len(old))]
        adds += len(old) - d                            # 第二道栅栏之后才写
        rows.append(temp[:])
        d *= 2
    return rows, adds

**函数 `inclusive`**（来自 `ch12_scan.py`）：v_j = 前 j 个（含自己）之和。

In [ ]:
def inclusive(v) -> np.ndarray:
    """v_j = 前 j 个（含自己）之和。"""
    return np.cumsum(v)

**函数 `three_phase_scan`**（来自 `ch12_scan.py`）：跨 block：① 每个 block 局部 scan 并写出总和；② 对 block 总和做 exclusive scan；③ 每个 block 加回前面的总和。

In [ ]:
def three_phase_scan(v: np.ndarray, block: int) -> np.ndarray:
    """跨 block：① 每个 block 局部 scan 并写出总和；② 对 block 总和做 exclusive scan；③ 每个 block 加回前面的总和。"""
    parts = v.reshape(-1, block)
    local = np.cumsum(parts, axis=1)              # kernel ①
    offsets = exclusive(local[:, -1])             # kernel ②
    return (local + offsets[:, None]).ravel()

**函数 `shfl_up`**（来自 `ch11_shuffle_atomics.py`）：lane t 拿 lane t − d 的值；t < d 的拿回自己的值。

In [ ]:
def shfl_up(v: np.ndarray, d: int) -> np.ndarray:
    """lane t 拿 lane t − d 的值；t < d 的拿回自己的值。"""
    width = len(v)
    src = np.arange(width) - d
    src[src < 0] = np.arange(width)[src < 0]
    return v[src]

**函数 `warp_scan`**（来自 `ch12_scan.py`）：warp 内 inclusive scan：for d = 1, 2, 4…：t = shfl_up(v, d)；lane ≥ d 的 v += t。

In [ ]:
def warp_scan(v: np.ndarray) -> np.ndarray:
    """warp 内 inclusive scan：for d = 1, 2, 4…：t = shfl_up(v, d)；lane ≥ d 的 v += t。"""
    v = v.copy()
    lane = np.arange(len(v))
    d = 1
    while d < len(v):
        t = shfl_up(v, d)
        v = np.where(lane >= d, v + t, v)
        d *= 2
    return v

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
v = [3, 1, 4, 1, 5, 9, 2, 6]

In [ ]:
print("\n第一步 inclusive / exclusive")
print("  inclusive:", inclusive(v))
print("  exclusive:", exclusive(v))

In [ ]:
print("\n第二步 Hillis–Steele")
rows, adds = hillis_steele(v)
for d, row in zip((1, 2, 4), rows):
    print(f"  d = {d}: {row}")
print("  加法次数 =", adds, "（顺序算法只要", len(v) - 1, "次）")

In [ ]:
print("\n第三步 Blelloch")
ex, adds = blelloch(v)
print("  结果（exclusive）:", ex, "；加法次数 =", adds)

In [ ]:
print("\n第四步 warp 内用 __shfl_up_sync（8 个 lane）")
print("  ", warp_scan(np.array(v)))

In [ ]:
print("\n第五步 三段式：32 个数分成 4 个 block，每个 8 个")
rng = np.random.default_rng(0)
big = rng.integers(0, 10, 32)
print("  和 np.cumsum 一样：", np.array_equal(three_phase_scan(big, 8), np.cumsum(big)))

## 3. 代码里才看得到的

1. **`old = temp[:]` 就是"读完再写"**：Hillis–Steele 要先把整行旧值拷一份再算新值；如果原地改，后面的线程会读到这一轮已经改过的数——讲义里第二道 `__syncthreads()` 防的就是这个。
2. **加法次数按真正做了加法的线程数**：Hillis–Steele 第 d 轮只有 t ≥ d 的线程在加，8 个数是 7 + 6 + 4 = 17 次。

## 4. 自己改一改

- 把输入换成 16 个数，比较 Hillis–Steele 和 Blelloch 的加法次数，看差距怎么随 N 变大。
- 把 `three_phase_scan` 的 block 大小换成 4、16。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> scan：每个位置要前面所有数的和；inclusive 带自己，exclusive 不带；
> Hillis–Steele 每轮看远一倍：log N 轮，N log N 次加，适合 warp 里；
> Blelloch 先上树后下树：2 log N 轮，2N 次加，省工作量；
> 读完再写，两道栅栏；
> 跨 block：三段式最稳，单趟靠原子发号 + 往回看，生产用 CUB。